In [ ]:
"""
Project 2 Week 4: Business Dashboard
======================================
Member 3: Creates interactive visualizations using
Plotly and Matplotlib showing Revenue Trend,
Occupancy Rate, Reward Trend, and Pricing Distribution.

Author  : Varnika Valliammai V
File    : business_dashboard.ipynb
"""

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


# Load training data

# Load DQN training history
try:
    dqn_history = pd.read_csv('dqn_training_history.csv')
    print("DQN history loaded:", dqn_history.shape)
    print(dqn_history.head(3))
except FileNotFoundError:
    # Generate sample data if file not available yet
    print("DQN history not found — generating sample data...")
    np.random.seed(42)
    n = 500
    rewards = np.cumsum(np.random.randn(n) * 50 + 200) / np.arange(1, n+1) * 100
    dqn_history = pd.DataFrame({
        'episode'          : range(n),
        'total_reward'     : rewards + np.random.randn(n) * 30,
        'avg_loss'         : np.random.exponential(0.1, n),
        'cumulative_reward': np.cumsum(rewards),
        'avg_reward_50'    : pd.Series(rewards).rolling(50, min_periods=1).mean()
    })
    print("Sample data generated for dashboard testing.")

# Load step logs (daily booking details)
try:
    step_logs = pd.read_csv('step_logs.csv')
    print("Step logs loaded:", step_logs.shape)
except FileNotFoundError:
    print("Step logs not found — generating sample data...")
    np.random.seed(42)
    n_steps = 500 * 30
    step_logs = pd.DataFrame({
        'episode'   : np.repeat(range(500), 30),
        'day'       : np.tile(range(1, 31), 500),
        'inventory' : np.random.randint(0, 100, n_steps),
        'days_left' : np.tile(range(30, 0, -1), 500),
        'action'    : np.random.randint(0, 5, n_steps),
        'price'     : np.random.choice([50,100,150,200,250], n_steps),
        'bookings'  : np.random.randint(0, 6, n_steps),
        'reward'    : np.random.uniform(0, 500, n_steps)
    })
    print("Sample step data generated.")

In [ ]:

# Dashboard 1: Revenue Trend


fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=(
        'Episode Revenue Over Training',
        'Rolling Average Revenue (50 episodes)'
    ),
    vertical_spacing=0.15
)

# Episode revenue
fig.add_trace(
    go.Scatter(
        x    = dqn_history['episode'],
        y    = dqn_history['total_reward'],
        mode = 'lines',
        name = 'Episode Revenue',
        line = dict(color='steelblue', width=1),
        opacity=0.6
    ),
    row=1, col=1
)

# Rolling average
fig.add_trace(
    go.Scatter(
        x    = dqn_history['episode'],
        y    = dqn_history['avg_reward_50'],
        mode = 'lines',
        name = 'Rolling Avg (50)',
        line = dict(color='orange', width=2.5)
    ),
    row=2, col=1
)

fig.update_layout(
    title_text = '📈 Revenue Trend — DQN Agent Training',
    height     = 600,
    template   = 'plotly_white',
    showlegend = True
)

fig.update_xaxes(title_text='Episode', row=1, col=1)
fig.update_xaxes(title_text='Episode', row=2, col=1)
fig.update_yaxes(title_text='Revenue (₹)', row=1, col=1)
fig.update_yaxes(title_text='Avg Revenue (₹)', row=2, col=1)

fig.show()
print(f"Peak revenue    : ₹{dqn_history['total_reward'].max():,.2f}")
print(f"Average revenue : ₹{dqn_history['total_reward'].mean():,.2f}")
print(f"Final avg (50)  : ₹{dqn_history['avg_reward_50'].iloc[-1]:,.2f}")

In [ ]:

# Dashboard 2: Occupancy Rate

# Calculate occupancy per episode
occupancy = step_logs.groupby('episode').agg(
    total_booked = ('bookings', 'sum')
).reset_index()

occupancy['occupancy_rate'] = (
    occupancy['total_booked'] / 100 * 100
).clip(0, 100)

occupancy['occupancy_roll'] = (
    occupancy['occupancy_rate']
    .rolling(50, min_periods=1).mean()
)

fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Occupancy Rate Over Training',
        'Occupancy Rate Distribution'
    )
)

# Line chart
fig2.add_trace(
    go.Scatter(
        x    = occupancy['episode'],
        y    = occupancy['occupancy_rate'],
        mode = 'lines',
        name = 'Occupancy %',
        line = dict(color='green', width=1),
        opacity=0.5
    ),
    row=1, col=1
)

fig2.add_trace(
    go.Scatter(
        x    = occupancy['episode'],
        y    = occupancy['occupancy_roll'],
        mode = 'lines',
        name = 'Rolling Avg',
        line = dict(color='darkgreen', width=2.5)
    ),
    row=1, col=1
)

# Histogram
fig2.add_trace(
    go.Histogram(
        x    = occupancy['occupancy_rate'],
        name = 'Distribution',
        marker_color = 'green',
        opacity=0.7
    ),
    row=1, col=2
)

fig2.update_layout(
    title_text = '🏨 Occupancy Rate Analysis',
    height     = 450,
    template   = 'plotly_white'
)

fig2.update_xaxes(title_text='Episode',     row=1, col=1)
fig2.update_xaxes(title_text='Occupancy %', row=1, col=2)
fig2.update_yaxes(title_text='Occupancy %', row=1, col=1)
fig2.update_yaxes(title_text='Count',       row=1, col=2)

fig2.show()
print(f"Average occupancy : {occupancy['occupancy_rate'].mean():.1f}%")
print(f"Peak occupancy    : {occupancy['occupancy_rate'].max():.1f}%")

In [ ]:

# Dashboard 3: Reward Trend + Loss


fig3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Cumulative Reward Over Training',
        'Training Loss Over Episodes'
    )
)

# Cumulative reward
fig3.add_trace(
    go.Scatter(
        x    = dqn_history['episode'],
        y    = dqn_history['cumulative_reward'],
        mode = 'lines',
        name = 'Cumulative Reward',
        line = dict(color='purple', width=2),
        fill = 'tozeroy',
        fillcolor = 'rgba(128,0,128,0.1)'
    ),
    row=1, col=1
)

# Training loss
fig3.add_trace(
    go.Scatter(
        x    = dqn_history['episode'],
        y    = dqn_history['avg_loss'],
        mode = 'lines',
        name = 'Avg Loss',
        line = dict(color='red', width=1.5)
    ),
    row=1, col=2
)

fig3.update_layout(
    title_text = '📊 Reward Trend & Training Loss',
    height     = 450,
    template   = 'plotly_white'
)

fig3.update_xaxes(title_text='Episode', row=1, col=1)
fig3.update_xaxes(title_text='Episode', row=1, col=2)
fig3.update_yaxes(title_text='Cumulative Revenue (₹)', row=1, col=1)
fig3.update_yaxes(title_text='MSE Loss', row=1, col=2)

fig3.show()
print(f"Total cumulative revenue : ₹{dqn_history['cumulative_reward'].iloc[-1]:,.2f}")
print(f"Final avg loss           : {dqn_history['avg_loss'].iloc[-1]:.4f}")

In [ ]:

# Dashboard 4: Pricing Distribution


price_counts = step_logs['price'].value_counts().sort_index()
price_labels = [f'₹{p}' for p in price_counts.index]

# Pie chart — overall pricing distribution
fig4 = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Overall Pricing Distribution',
        'Price vs Average Revenue'
    ),
    specs=[[{'type': 'pie'}, {'type': 'bar'}]]
)

fig4.add_trace(
    go.Pie(
        labels    = price_labels,
        values    = price_counts.values,
        name      = 'Pricing Mix',
        hole      = 0.3,
        marker    = dict(colors=[
            '#636EFA', '#EF553B', '#00CC96',
            '#AB63FA', '#FFA15A'
        ])
    ),
    row=1, col=1
)

# Average revenue per price level
avg_rev_by_price = step_logs.groupby('price')['reward'].mean()

fig4.add_trace(
    go.Bar(
        x    = [f'₹{p}' for p in avg_rev_by_price.index],
        y    = avg_rev_by_price.values,
        name = 'Avg Revenue',
        marker_color = '#00CC96'
    ),
    row=1, col=2
)

fig4.update_layout(
    title_text = '💰 Pricing Distribution & Revenue per Price Level',
    height     = 450,
    template   = 'plotly_white'
)

fig4.update_xaxes(title_text='Price Level', row=1, col=2)
fig4.update_yaxes(title_text='Avg Revenue (₹)', row=1, col=2)

fig4.show()

print("Price distribution:")
for price, count in zip(price_counts.index, price_counts.values):
    pct = count / price_counts.sum() * 100
    print(f"  ₹{price}: {count} times ({pct:.1f}%)")

In [ ]:

# Combined Full Dashboard


fig_full = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        '📈 Revenue Trend',
        '🏨 Occupancy Rate',
        '📊 Cumulative Reward',
        '💰 Pricing Distribution'
    ),
    specs=[
        [{'type': 'scatter'}, {'type': 'scatter'}],
        [{'type': 'scatter'}, {'type': 'bar'}]
    ],
    vertical_spacing   = 0.15,
    horizontal_spacing = 0.10
)

# Revenue trend
fig_full.add_trace(
    go.Scatter(
        x    = dqn_history['episode'],
        y    = dqn_history['avg_reward_50'],
        mode = 'lines',
        name = 'Revenue Trend',
        line = dict(color='steelblue', width=2)
    ),
    row=1, col=1
)

# Occupancy
fig_full.add_trace(
    go.Scatter(
        x    = occupancy['episode'],
        y    = occupancy['occupancy_roll'],
        mode = 'lines',
        name = 'Occupancy %',
        line = dict(color='green', width=2)
    ),
    row=1, col=2
)

# Cumulative reward
fig_full.add_trace(
    go.Scatter(
        x    = dqn_history['episode'],
        y    = dqn_history['cumulative_reward'],
        mode = 'lines',
        name = 'Cumulative Revenue',
        line = dict(color='purple', width=2),
        fill = 'tozeroy',
        fillcolor = 'rgba(128,0,128,0.1)'
    ),
    row=2, col=1
)

# Pricing distribution
fig_full.add_trace(
    go.Bar(
        x    = [f'₹{p}' for p in avg_rev_by_price.index],
        y    = avg_rev_by_price.values,
        name = 'Avg Rev per Price',
        marker_color = '#FFA15A'
    ),
    row=2, col=2
)

fig_full.update_layout(
    title_text = '🏨 Dynamic Pricing — Business Dashboard',
    height     = 700,
    template   = 'plotly_white',
    showlegend = False
)

fig_full.show()

print("\n" + "=" * 50)
print("BUSINESS DASHBOARD SUMMARY")
print("=" * 50)
print(f"Total episodes trained  : {len(dqn_history)}")
print(f"Average revenue/episode : ₹{dqn_history['total_reward'].mean():,.2f}")
print(f"Best episode revenue    : ₹{dqn_history['total_reward'].max():,.2f}")
print(f"Average occupancy rate  : {occupancy['occupancy_rate'].mean():.1f}%")
print(f"Total cumulative revenue: ₹{dqn_history['cumulative_reward'].iloc[-1]:,.2f}")
print(f"Most used price         : ₹{step_logs['price'].mode()[0]}")
print("=" * 50)